### Build a Simple LLM Application with LCEL
In this quickstart we'll show you how to build a simple LLM application with LangChain. This application will translate text from English into another language. This is a relatively simple LLM application - it's just a single LLM call plus some prompting. Still, this is a great way to get started with LangChain - a lot of features can be built with just some prompting and an LLM call!

After seeing this video, you'll have a high level overview of:

- Using language models

- Using PromptTemplates and OutputParsers

- Using LangChain Expression Language (LCEL) to chain components together

- Debugging and tracing your application using LangSmith

- Deploying your application with LangServe

In [13]:
import requests

response = requests.get("https://www.google.com", timeout=10,verify=False)
print(response.status_code)

d:\MindsprintCourse\GenAI_Apps\simplellmLECL\venv\lib\site-packages\urllib3\connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'www.google.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


200


In [2]:
import os
from dotenv import load_dotenv
load_dotenv()

True

In [3]:
groq_api_key = os.getenv("GROQ_API_KEY")

In [ ]:
from langchain_groq import ChatGroq

model = ChatGroq(
    model="openai/gpt-oss-120b",
    groq_api_key=groq_api_key,
    
)
model

ChatGroq(profile={'max_input_tokens': 131072, 'max_output_tokens': 32768, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x00000178CB222D10>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x00000178CB222C20>, model_name='openai/gpt-oss-120b', model_kwargs={}, groq_api_key=SecretStr('**********'))

In [6]:
from langchain_core.messages import HumanMessage,SystemMessage

# SystemMessage instruct the llm how to behave
message = [
    SystemMessage(content="Translate the following message from english to hindi."),
    HumanMessage(content="Hello, How are you?")
]
response = model.invoke(message)

In [7]:
print(response)

content='नमस्ते, आप कैसे हैं?' additional_kwargs={'reasoning_content': 'The user asks: "Translate the following message from english to hindi." Then they provide "Hello, How are you?" So answer should be the Hindi translation. Possibly also include original? Likely just translation. Provide "नमस्ते, आप कैसे हैं?" Ensure correct punctuation.'} response_metadata={'token_usage': {'completion_tokens': 76, 'prompt_tokens': 89, 'total_tokens': 165, 'completion_time': 0.159855192, 'completion_tokens_details': {'reasoning_tokens': 58}, 'prompt_time': 0.003437363, 'prompt_tokens_details': None, 'queue_time': 0.044589657, 'total_time': 0.163292555}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_d29d1d1418', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'} id='lc_run--019d2e6f-9c65-76f3-892f-3ab75f263b81-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 89, 'output_tokens': 76, 'total_tokens': 165, 'output_toke

In [8]:
# above is ai message we only want ai message content or translate 
from langchain_core.output_parsers import StrOutputParser
parser = StrOutputParser()
parser.invoke(response)

'नमस्ते, आप कैसे हैं?'

In [16]:
## using LCEL -> we can chain the components
chain = model|parser
chain.invoke(message)

'नमस्ते, आप कैसे हैं?'

In [17]:
## instead of giving list of message we can use prompt templates

from langchain_core.prompts import ChatPromptTemplate
generic_template = "Translate the following in the {language}: "

prompt = ChatPromptTemplate.from_messages(
    [
        ("system",generic_template),
        ("user","{text}")
    ]
)

In [25]:
result = prompt.invoke(
    {
        "language":"Hindi",
        "text":"What is your name"
    }
)

In [26]:
result.messages

[SystemMessage(content='Translate the following in the Hindi: ', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='What is your name', additional_kwargs={}, response_metadata={})]

In [29]:
result.to_messages()

[SystemMessage(content='Translate the following in the Hindi: ', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='What is your name', additional_kwargs={}, response_metadata={})]

In [30]:
print(result.to_messages())

[SystemMessage(content='Translate the following in the Hindi: ', additional_kwargs={}, response_metadata={}), HumanMessage(content='What is your name', additional_kwargs={}, response_metadata={})]


In [ ]:
# chaining together component with LECL
chain = prompt|model|parser

In [32]:
chain

ChatPromptTemplate(input_variables=['language', 'text'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=['language'], input_types={}, partial_variables={}, template='Translate the following in the {language}: '), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['text'], input_types={}, partial_variables={}, template='{text}'), additional_kwargs={})])
| ChatGroq(profile={'max_input_tokens': 131072, 'max_output_tokens': 32768, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x00000178CB222D10>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x00000178CB222C20>, model_name='openai/gpt-oss-120b', model_kwargs={}, groq_api_key=SecretStr('**********'))
| StrOut

In [35]:
chain.invoke(
    {
        "language":"Hindi",
        "text":"My name is Pratham"
    }
)

'**मेरा नाम प्रथम है।**'